# GRU Prefetcher V10 -- reuse distance + cache/phase state + bypass gate

Controlled-variable sweep after V9. V10 keeps the GRU/delta-bitmap setup, then adds hardware-style utility features one at a time: PC, PC+delta, reuse distance, cache-set pressure, phase, and an optional suppress/bypass head.

Goal: do not only maximize accuracy/F1. Track latency, trigger rate, avg degree, and then replay IPC with ChampSim.


In [ ]:
import os, math, time, random, collections
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0); random.seed(0)
print('device',DEVICE)


In [ ]:
# ===== config =====
TRACE_CSV='/content/access_trace.605.mcf_s-994B.csv'  # change to 619.lbm_s-4268B / 602.gcc_s-734B
USE_L1DM_ONLY=True
LINE_BITS=6; HIST=16; DELTA_RANGE=64; NEXT_WINDOW=8
DELTA_VOCAB_K=2048; BITMAP_SIZE=2*DELTA_RANGE+1; VOCAB_SIZE=DELTA_VOCAB_K+1
NUM_PC=1024; NUM_PCD=4096; NUM_REUSE=16; NUM_PRESS=16; NUM_PHASE=16
NSETS_APPROX=2048; PRESSURE_WINDOW=4096; PHASE_LEN=16384
HIDDEN=64; EMB_D=32; EMB_PC=16; EMB_PCD=16; EMB_R=8; EMB_S=8
EPOCHS=6; BATCH=1024; LR=1e-3; WD=1e-5; CLIP=1.0
PROB_TH=0.35; SUPPRESS_TH=0.55; MAX_DEGREE=2
OUT_DIR='results/generated/prefetch_lists'
TRACE_TAG=Path(TRACE_CSV).name.replace('access_trace.','').replace('.csv','')
print('trace',TRACE_TAG)


In [ ]:
# ===== load + featurize =====
def parse_int(x):
    if isinstance(x,str):
        x=x.strip(); return int(x,16) if x.lower().startswith('0x') else int(x)
    return int(x)
def log_bucket(x,nb): return nb-1 if x is None or x<0 else min(nb-1,int(math.log2(x+1)))
def h2(a,b,mod): return ((int(a)*1315423911)^(int(b)*2654435761))%mod
df=pd.read_csv(TRACE_CSV).dropna(subset=['addr_hex','pc_hex','hit']).reset_index(drop=True)
df['addr']=df.addr_hex.apply(parse_int).astype('int64'); df['pc']=df.pc_hex.apply(parse_int).astype('int64'); df['hit']=df.hit.astype('int8')
df['idx_orig']=df['idx'].astype('int64') if 'idx' in df.columns else np.arange(len(df),dtype=np.int64)
if USE_L1DM_ONLY: df=df[df.hit==0].reset_index(drop=True)
N=len(df); addrs=df.addr.values.astype(np.int64); lines=addrs>>LINE_BITS; pcs=df.pc.values.astype(np.int64); idxs=df.idx_orig.values.astype(np.int64)
i_tr=int(.70*N); i_va=int(.85*N); valid_end=N-NEXT_WINDOW
print(f'rows={N:,} split train={i_tr:,} val={i_va-i_tr:,} test={valid_end-i_va:,}')
last_pc={}; deltas=np.zeros(N,dtype=np.int64)
for i,(pc,line) in enumerate(zip(pcs,lines)):
    deltas[i]=line-last_pc[pc] if pc in last_pc else 0; last_pc[pc]=line
cnt=collections.Counter(map(int,deltas[1:i_tr])); top=[d for d,_ in cnt.most_common(DELTA_VOCAB_K)]
d2id={d:i+1 for i,d in enumerate(top)}; cov=sum(cnt[d] for d in top)/max(1,sum(cnt.values()))
print('vocab coverage',round(100*cov,1),'top10',top[:10])
Xd=np.zeros((N,HIST),np.int64); Xpc=np.zeros(N,np.int64); Xpcd=np.zeros(N,np.int64); Xr=np.zeros(N,np.int64); Xp=np.zeros(N,np.int64); Xph=np.zeros(N,np.int64)
Y=np.zeros((N,BITMAP_SIZE),np.float32); Ys=np.zeros(N,np.float32)
hist=collections.defaultdict(lambda:collections.deque([0]*HIST,maxlen=HIST)); last_line={}; setcnt=collections.Counter(); setw=collections.deque(); C=DELTA_RANGE
t=time.time()
for i in range(N):
    pc=int(pcs[i]); line=int(lines[i]); d=int(deltas[i])
    Xd[i]=np.fromiter(hist[pc],dtype=np.int64,count=HIST); Xpc[i]=pc%NUM_PC; Xpcd[i]=h2(pc,d,NUM_PCD)
    Xr[i]=log_bucket(i-last_line[line],NUM_REUSE) if line in last_line else NUM_REUSE-1
    sid=line%NSETS_APPROX; Xp[i]=min(NUM_PRESS-1,int(math.log2(setcnt[sid]+1))); Xph[i]=(i//PHASE_LEN)%NUM_PHASE
    ok=False
    for j in range(i+1,min(N,i+1+NEXT_WINDOW)):
        fd=int(lines[j]-line)
        if -DELTA_RANGE<=fd<=DELTA_RANGE: Y[i,fd+C]=1.; ok=True
    Ys[i]=0. if ok else 1.
    hist[pc].append(d2id.get(d,0)); last_line[line]=i; setw.append(sid); setcnt[sid]+=1
    if len(setw)>PRESSURE_WINDOW:
        old=setw.popleft(); setcnt[old]-=1
print('features sec',round(time.time()-t,1),'density',float(Y[:valid_end].mean()))


In [ ]:
# ===== dataset/model/sweep =====
class DS(Dataset):
    def __init__(self,lo,hi): self.lo=lo; self.hi=min(hi,valid_end)
    def __len__(self): return self.hi-self.lo
    def __getitem__(self,k):
        i=self.lo+k
        return tuple(torch.tensor(x,dtype=torch.long) for x in (Xd[i],Xpc[i],Xpcd[i],Xr[i],Xp[i],Xph[i]))+(torch.tensor(Y[i]),torch.tensor(Ys[i]))
class M(nn.Module):
    def __init__(self,pc=0,pcd=0,reuse=0,state=0,bypass=0):
        super().__init__(); self.pc=pc; self.pcd=pcd; self.reuse=reuse; self.state=state; self.bypass=bypass
        self.ed=nn.Embedding(VOCAB_SIZE,EMB_D,padding_idx=0); self.gru=nn.GRU(EMB_D,HIDDEN,batch_first=True); extra=HIDDEN
        if pc: self.epc=nn.Embedding(NUM_PC,EMB_PC); extra+=EMB_PC
        if pcd: self.epcd=nn.Embedding(NUM_PCD,EMB_PCD); extra+=EMB_PCD
        if reuse: self.er=nn.Embedding(NUM_REUSE,EMB_R); extra+=EMB_R
        if state: self.epr=nn.Embedding(NUM_PRESS,EMB_S); self.eph=nn.Embedding(NUM_PHASE,EMB_S); extra+=2*EMB_S
        self.trunk=nn.Sequential(nn.Linear(extra,128),nn.ReLU(),nn.Dropout(.1),nn.Linear(128,96),nn.ReLU())
        self.bm=nn.Linear(96,BITMAP_SIZE)
        if bypass: self.sup=nn.Linear(96,1)
    def forward(self,xd,xpc,xpcd,xr,xp,xph):
        _,h=self.gru(self.ed(xd)); parts=[h[-1]]
        if self.pc: parts.append(self.epc(xpc))
        if self.pcd: parts.append(self.epcd(xpcd))
        if self.reuse: parts.append(self.er(xr))
        if self.state: parts += [self.epr(xp),self.eph(xph)]
        z=self.trunk(torch.cat(parts,1)); return self.bm(z), (self.sup(z).squeeze(1) if self.bypass else None)
variants={'V10a_delta':{},'V10b_pc':{'pc':1},'V10c_pcd':{'pc':1,'pcd':1},'V10d_reuse':{'pc':1,'pcd':1,'reuse':1},'V10e_state':{'pc':1,'pcd':1,'reuse':1,'state':1},'V10f_bypass':{'pc':1,'pcd':1,'reuse':1,'state':1,'bypass':1}}
def loader(lo,hi,sh): return DataLoader(DS(lo,hi),batch_size=BATCH,shuffle=sh)
tr,va,te=loader(0,i_tr,1),loader(i_tr,i_va,0),loader(i_va,valid_end,0)
def evalm(model,ld):
    model.eval(); n=top1=top5=tp=fp=fn=0
    with torch.no_grad():
      for xd,xpc,xpcd,xr,xp,xph,y,ys in ld:
        xd,xpc,xpcd,xr,xp,xph=[t.to(DEVICE) for t in (xd,xpc,xpcd,xr,xp,xph)]; y=y.to(DEVICE)
        logits,sup=model(xd,xpc,xpcd,xr,xp,xph); p=torch.sigmoid(logits); pred=(p>=PROB_TH).float()
        if sup is not None: pred[torch.sigmoid(sup)>=SUPPRESS_TH]=0.
        top1+=(y.gather(1,p.topk(1,1).indices).max(1).values>.5).sum().item(); top5+=(y.gather(1,p.topk(5,1).indices).max(1).values>.5).sum().item()
        tp+=(pred*y).sum().item(); fp+=(pred*(1-y)).sum().item(); fn+=((1-pred)*y).sum().item(); n+=y.size(0)
    P=tp/max(1,tp+fp); R=tp/max(1,tp+fn); F=2*P*R/max(1e-9,P+R)
    return {'top1':top1/max(1,n),'top5':top5/max(1,n),'precision':P,'recall':R,'f1':F,'n':n}
rows=[]; best=None; best_score=-9; best_model=None
pos=float(Y[:i_tr].mean()); pw=torch.tensor([(1-pos)/max(pos,1e-6)],device=DEVICE)
for name,cfg in variants.items():
    print('\n===',name,cfg,'==='); model=M(**cfg).to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD)
    for ep in range(EPOCHS):
      model.train(); loss_sum=nb=0
      for xd,xpc,xpcd,xr,xp,xph,y,ys in tr:
        xd,xpc,xpcd,xr,xp,xph=[t.to(DEVICE) for t in (xd,xpc,xpcd,xr,xp,xph)]; y=y.to(DEVICE); ys=ys.to(DEVICE)
        bm,sup=model(xd,xpc,xpcd,xr,xp,xph); loss=F.binary_cross_entropy_with_logits(bm,y,pos_weight=pw)
        if sup is not None: loss=loss+.25*F.binary_cross_entropy_with_logits(sup,ys)
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),CLIP); opt.step(); loss_sum+=loss.item(); nb+=1
      vm=evalm(model,va); print(ep+1,'loss',round(loss_sum/max(1,nb),4),'val_f1',round(vm['f1'],3),'val_top1',round(vm['top1'],3))
    tm=evalm(model,te); params=sum(p.numel() for p in model.parameters()); rows.append({'variant':name,'params':params,**tm})
    score=tm['f1']-.0000005*params
    if score>best_score: best_score=score; best=name; best_model=model
s=pd.DataFrame(rows); s.to_csv(f'gru_v10_summary_{TRACE_TAG}.csv',index=False); display(s); print('best',best)


In [ ]:
# ===== export best prefetch list =====
os.makedirs(OUT_DIR,exist_ok=True)
out=f'{OUT_DIR}/prefetch_list_GRU_V10_{best}_{TRACE_TAG}.txt'
best_model.eval(); n_emit=n_acc=n_sup=0; C=DELTA_RANGE
with open(out,'w') as fh, torch.no_grad():
  ld=DataLoader(DS(i_va,valid_end),batch_size=4096,shuffle=False)
  for bi,b in enumerate(ld):
    xd,xpc,xpcd,xr,xp,xph,y,ys=b; xd,xpc,xpcd,xr,xp,xph=[t.to(DEVICE) for t in (xd,xpc,xpcd,xr,xp,xph)]
    bm,sup=best_model(xd,xpc,xpcd,xr,xp,xph); prob=torch.sigmoid(bm).cpu().numpy(); suppress=np.zeros(prob.shape[0],bool)
    if sup is not None: suppress=(torch.sigmoid(sup).cpu().numpy()>=SUPPRESS_TH)
    for j,row in enumerate(prob):
      gi=i_va+bi*4096+j; n_acc+=1
      if suppress[j]: n_sup+=1; continue
      for ti in np.argsort(-row)[:MAX_DEGREE]:
        if row[ti]<PROB_TH: break
        d=int(ti)-C
        if d==0: continue
        pf=(int(addrs[gi])+(d<<LINE_BITS)) & 0xffffffffffffffff
        fh.write(f'{int(idxs[gi])} 0x{pf:x}\n'); n_emit+=1
print('wrote',out,'accesses',n_acc,'prefetches',n_emit,'avg_degree',round(n_emit/max(1,n_acc),2),'suppress%',round(100*n_sup/max(1,n_acc),1))
print(f'TRACE={TRACE_TAG} PFETCH=$PWD/{out} MODEL_TAG=GRU_V10_{best}_{TRACE_TAG} bash scripts/run_nn_replay.sh')
